# 03 — Activation patching

Do Notebook 02 recebemos heads candidatas e uma distinção importante: DLA mede alinhamento direto com a resposta, não necessidade causal. Agora faremos uma intervenção:

1. executamos um prompt **limpo** e guardamos suas ativações;
2. executamos a versão **corrompida**;
3. substituímos uma ativação corrompida pela ativação limpa correspondente;
4. medimos quanto do comportamento correto foi recuperado.

Os pesos do modelo não mudam. O patch é uma substituição temporária durante uma execução. Os mapas produzidos aqui localizarão ativações causais; o Notebook 04 restringirá essas intervenções a caminhos entre heads.

In [1]:
# Mantemos modelo, prompts e respostas; muda apenas a intervenção.
from functools import partial

import plotly.graph_objects as go
import torch
from transformer_lens import patching
import transformer_lens.utilities as utils

from ioi_replication import (
    build_ioi_examples,
    load_gpt2_small,
    logits_to_logit_diff,
    normalized_recovery,
    select_device,
    tokenize_answers,
)

device = select_device()
model = load_gpt2_small(device)
prompts, answers = build_ioi_examples()
# `tokens` é (batch, position); cada linha de `answer_tokens` contém [correto, incorreto].
tokens = model.to_tokens(prompts, prepend_bos=True).to(device)
answer_tokens = tokenize_answers(model, answers, device)

# Em cada par adjacente, trocamos os papéis IO/S: o nome correto vira o sujeito repetido e vice-versa.
swap_indices = torch.tensor(
    [i + 1 if i % 2 == 0 else i - 1 for i in range(len(tokens))],
    device=device,
)
corrupted_tokens = tokens[swap_indices]
print(f"Dispositivo: {device}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Dispositivo: cuda


In [2]:
# Conferimos a troca IO/S criada por `swap_indices` antes de usá-la como corrupção causal.
for clean_index, corrupted_index in enumerate(swap_indices.tolist()):
    print(f"limpo {clean_index}:      {prompts[clean_index]}")
    print(f"corrompido {clean_index}: {prompts[corrupted_index]}\n")

# O cache limpo fornece as ativações a transplantar; da execução corrompida bastam os logits.
with torch.no_grad():
    clean_logits, clean_cache = model.run_with_cache(tokens)
    corrupted_logits = model(corrupted_tokens)

# Como avaliamos os tokens de resposta do prompt limpo após trocar IO e S, o baseline corrompido inverte o sinal.
clean_diff = logits_to_logit_diff(clean_logits, answer_tokens)
corrupted_diff = logits_to_logit_diff(corrupted_logits, answer_tokens)
print(f"logit diff limpo:      {clean_diff.item():+.3f}")
print(f"logit diff corrompido: {corrupted_diff.item():+.3f}")

limpo 0:      When John and Mary went to the shops, John gave the bag to
corrompido 0: When John and Mary went to the shops, Mary gave the bag to

limpo 1:      When John and Mary went to the shops, Mary gave the bag to
corrompido 1: When John and Mary went to the shops, John gave the bag to

limpo 2:      When Tom and James went to the park, James gave the ball to
corrompido 2: When Tom and James went to the park, Tom gave the ball to

limpo 3:      When Tom and James went to the park, Tom gave the ball to
corrompido 3: When Tom and James went to the park, James gave the ball to

limpo 4:      When Dan and Sid went to the shops, Sid gave an apple to
corrompido 4: When Dan and Sid went to the shops, Dan gave an apple to

limpo 5:      When Dan and Sid went to the shops, Dan gave an apple to
corrompido 5: When Dan and Sid went to the shops, Sid gave an apple to

limpo 6:      After Martin and Amy went to the park, Amy gave a drink to
corrompido 6: After Martin and Amy went to the park, 

logit diff limpo:      +3.552
logit diff corrompido: -3.552


## Recuperação normalizada

Mediremos o resultado do patch em uma escala comum:

$$\text{recuperação}=\frac{\text{patched}-\text{corrompido}}{\text{limpo}-\text{corrompido}}$$

- `0`: resultado igual ao baseline corrompido;
- `1`: resultado igual ao baseline limpo;
- valor negativo: o patch piorou o resultado em relação ao corrompido;
- valor acima de `1`: o patch superou o baseline limpo.

In [3]:
# A métrica fecha sobre respostas e baselines já calculados; cada execução patched fornece apenas novos logits.
def ioi_metric(logits: torch.Tensor) -> torch.Tensor:
    patched_diff = logits_to_logit_diff(logits, answer_tokens)
    return normalized_recovery(patched_diff, clean_diff, corrupted_diff)


# Verificamos os extremos e o ponto médio antes de usar a métrica em centenas de intervenções.
torch.testing.assert_close(ioi_metric(clean_logits), torch.tensor(1.0, device=device))
torch.testing.assert_close(ioi_metric(corrupted_logits), torch.tensor(0.0, device=device))
midpoint_logits = (clean_logits + corrupted_logits) / 2
# A recuperação 0.5 confirma a escala linear que será mostrada nos mapas seguintes.
torch.testing.assert_close(ioi_metric(midpoint_logits), torch.tensor(0.5, device=device))
print("Endpoints verificados: corrompido=0, meio=0.5, limpo=1")

Endpoints verificados: corrompido=0, meio=0.5, limpo=1


## 1. Patching do residual stream

Para cada combinação `(camada, posição)`, substituímos somente o `resid_pre` corrompido pela versão limpa.

O eixo vertical será a camada. O eixo horizontal será a posição do token. Cores mais altas indicam maior recuperação causal.

In [4]:
# A função recebe os tokens corrompidos, transplanta do `clean_cache` e avalia cada patch com `ioi_metric`.
with torch.no_grad():
    builtin_residual_results = patching.get_act_patch_resid_pre(
        model,
        corrupted_tokens,
        clean_cache,
        ioi_metric,
    )

# O mapa tem (layer, position): uma recuperação para cada ponto de entrada do residual stream.
assert builtin_residual_results.shape == (model.cfg.n_layers, tokens.shape[1])
position_labels = [
    f"{index}:{token!r}"
    for index, token in enumerate(model.to_str_tokens(tokens[0].detach().cpu()))
]

figure = go.Figure(
    data=go.Heatmap(
        z=builtin_residual_results.detach().cpu().numpy(),
        x=position_labels,
        y=[f"L{layer}" for layer in range(model.cfg.n_layers)],
        colorscale="RdBu",
        zmid=0,
        colorbar_title="recuperação",
        hovertemplate="camada=%{y}<br>posição=%{x}<br>recuperação=%{z:.3f}<extra></extra>",
    )
)
figure.update_layout(
    title="Patching de resid_pre por camada e posição",
    xaxis_title="posição do token no primeiro prompt",
    yaxis_title="camada",
    height=600,
)
figure.show()

# `argmax` é linear; `divmod` converte o índice achatado de volta para (layer, position).
best_flat = builtin_residual_results.argmax().item()
best_layer, best_position = divmod(best_flat, tokens.shape[1])
print(
    f"Maior recuperação: camada {best_layer}, posição {best_position}, "
    f"valor={builtin_residual_results.max().item():.3f}"
)

  0%|          | 0/180 [00:00<?, ?it/s]

Maior recuperação: camada 10, posição 14, valor=1.005


## 2. Implementação manual com hooks

A função pronta acima será nosso oráculo. Agora reproduziremos o mesmo experimento explicitamente:

- o hook intercepta `resid_pre`;
- troca apenas uma posição pela ativação do cache limpo;
- deixa o restante da execução corrompida inalterado.

Se entendemos a intervenção, os dois mapas devem coincidir.

In [5]:
# O hook recebe `resid_pre` com shape (batch, position, d_model) na camada em execução.
def patch_residual_position(
    corrupted_residual: torch.Tensor,
    hook,
    *,
    position: int,
) -> torch.Tensor:
    # Mantemos batch e dimensão residual inteiros, trocando apenas a posição escolhida pelo cache limpo.
    corrupted_residual[:, position, :] = clean_cache[hook.name][:, position, :]
    return corrupted_residual


# Preenchemos o mesmo mapa (layer, position) da função pronta, agora uma intervenção por execução.
own_residual_results = torch.zeros_like(builtin_residual_results)
with torch.no_grad():
    for layer in range(model.cfg.n_layers):
        hook_name = utils.get_act_name("resid_pre", layer)
        for position in range(tokens.shape[1]):
            patched_logits = model.run_with_hooks(
                corrupted_tokens,
                fwd_hooks=[
                    (
                        hook_name,
                        partial(
                            patch_residual_position,
                            position=position,
                        ),
                    )
                ],
            )
            own_residual_results[layer, position] = ioi_metric(patched_logits)

# A concordância valida nossa leitura da indexação antes do patching por head.
torch.testing.assert_close(
    own_residual_results,
    builtin_residual_results,
    rtol=1e-4,
    atol=1e-4,
)
max_error = (own_residual_results - builtin_residual_results).abs().max()
print(f"Implementação manual coincide com a função pronta; erro máximo={max_error.item():.2e}")

Implementação manual coincide com a função pronta; erro máximo=0.00e+00


## 3. Patching de heads individuais

Agora fazemos o patch **all-pos** da saída `z`: restauramos uma head em todas as posições de uma vez, mantendo as demais heads corrompidas. Por isso o resultado tem apenas os eixos `(camada, head)`.

Esta leitura complementa o DLA do Notebook 02, mas a célula abaixo mostra somente recovery. DLA alto não garante recovery alto: a intervenção restaura uma head isolada em uma execução ainda corrompida, e caminhos paralelos ou compensações podem reduzir seu efeito observado.

In [6]:
# Depois de validar o patch residual, restauramos a saída `z` limpa de uma head por vez, em todas as posições.
with torch.no_grad():
    head_results = patching.get_act_patch_attn_head_out_all_pos(
        model,
        corrupted_tokens,
        clean_cache,
        ioi_metric,
    )

# O resultado organiza a recuperação por (layer, head), sem eixo de posição porque ele foi patched por inteiro.
assert head_results.shape == (model.cfg.n_layers, model.cfg.n_heads)

figure = go.Figure(
    data=go.Heatmap(
        z=head_results.detach().cpu().numpy(),
        x=[f"H{head}" for head in range(model.cfg.n_heads)],
        y=[f"L{layer}" for layer in range(model.cfg.n_layers)],
        colorscale="RdBu",
        zmid=0,
        colorbar_title="recuperação",
        hovertemplate="camada=%{y}<br>head=%{x}<br>recuperação=%{z:.3f}<extra></extra>",
    )
)
figure.update_layout(
    title="Recuperação ao aplicar patch em cada head",
    xaxis_title="head",
    yaxis_title="camada",
    height=600,
)
figure.show()

# Novamente, `divmod` converte o índice linear do máximo em (layer, head).
best_flat = head_results.argmax().item()
best_layer, best_head = divmod(best_flat, model.cfg.n_heads)
print(
    f"Maior recuperação individual: head {best_layer}.{best_head}, "
    f"valor={head_results.max().item():.3f}"
)

  0%|          | 0/144 [00:00<?, ?it/s]

Maior recuperação individual: head 5.5, valor=0.343


In [7]:
# Reutilizamos famílias conhecidas; as Name Movers são as candidatas conferidas por DLA no Notebook 02.
candidate_groups = {
    "induction": [(5, 5), (5, 8), (5, 9), (6, 9)],
    "S-inhibition": [(7, 3), (7, 9), (8, 6), (8, 10)],
    "name mover": [(9, 6), (9, 9), (10, 0)],
}

# A célula imprime apenas recovery all-pos. Diferenças em relação ao DLA são esperadas pela intervenção isolada e por compensações.
for group, heads in candidate_groups.items():
    print(group)
    for layer, head in heads:
        print(f"  {layer}.{head}: recuperação={head_results[layer, head].item():+.3f}")
# O Notebook 04 tratará esses grupos como senders e receivers explícitos.

induction
  5.5: recuperação=+0.343
  5.8: recuperação=-0.005
  5.9: recuperação=+0.016
  6.9: recuperação=+0.130
S-inhibition
  7.3: recuperação=+0.124
  7.9: recuperação=+0.192
  8.6: recuperação=+0.319
  8.10: recuperação=+0.278
name mover
  9.6: recuperação=-0.048
  9.9: recuperação=+0.264
  10.0: recuperação=+0.084


## Conclusão

Activation patching fornece evidência causal localizada: o mapa residual mostra **onde** a informação limpa é útil, e o patch all-pos mostra quanto restaurar uma head inteira recupera sob esta intervenção.

Isso complementa o DLA do Notebook 02, mas as métricas não precisam concordar: DLA alto pode coexistir com recovery baixo quando a head é restaurada isoladamente e outros caminhos permanecem corrompidos ou compensam seu efeito. O Notebook 04 usará o mesmo par limpo/corrompido para isolar conexões sender → receiver e testar o circuito proposto como conjunto.